In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp
import requests
import pandas as pd
import io

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
import requests

version_url = "https://api.beta.ons.gov.uk/v1/datasets/cpih01/editions/time-series/versions"

response = requests.get(version_url)
response.raise_for_status()

versions = response.json()
latest_version = versions["items"][0]["version"]
print(f"Latest version: {latest_version}")


In [0]:
download_url = f"https://download.ons.gov.uk/downloads/datasets/cpih01/editions/time-series/versions/{latest_version}.csv"

response = requests.get(download_url)
response.raise_for_status()

df_pd = pd.read_csv(io.StringIO(response.text))
print(df_pd.shape)
print(df_pd.head())
print(df_pd.columns.tolist())

In [0]:
df_bronze = spark.createDataFrame(df_pd)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.cpih")
)

In [0]:
spark.sql("SELECT * FROM bronze.cpih LIMIT 5").show()
spark.sql("SELECT COUNT(*) FROM bronze.cpih").show()


In [0]:
# bronze.cpih loaded
# Source: ONS CPIH01 time series, version discovery via beta API
# 55,754 rows, all components and time periods
# Version 67 at time of load